# Citus Database Setup and SDK Demo (Single Node)

This notebook demonstrates how to set up a Citus database using a single-node docker container, populate the patch table with dummy data, and implement a Python SDK for interacting with the database. Worker node logic is omitted for single-node setup.

## 1. Install and Import Required Libraries

Install `psycopg` if not already installed, and import all required libraries for database interaction.

In [1]:
# Install psycopg if needed (uncomment if running in a new environment)
# !pip install psycopg[binary]

import psycopg
from psycopg.rows import dict_row
import random
import datetime
import base64
from db_client import CitusHeadClient
import os

In [2]:
# Import DB connection constants from constants.py
from constants import (
    CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD
)

In [3]:
# Set DB connection variables from constants (single-node)
DB_HOST = CITUS_HEAD_HOST
DB_PORT = CITUS_HEAD_PORT
DB_NAME = CITUS_HEAD_DB
DB_USER = CITUS_HEAD_USER
DB_PASSWORD = CITUS_HEAD_PASSWORD

In [4]:
NUM_PATCHES = 1000

## 0. Drop All Tables (Clean Start)

Drop all tables if they exist to ensure a clean setup.

In [5]:
# Drop all tables in reverse dependency order for a clean start
head_client = CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)
drop_sql = [
    "DROP TABLE IF EXISTS confusion_matrix_ln CASCADE;",
    "DROP TABLE IF EXISTS pred_patch_latest CASCADE;",
    "DROP TABLE IF EXISTS pred_patch_last CASCADE;",
    "DROP TABLE IF EXISTS patch CASCADE;",
    "DROP TABLE IF EXISTS label_class CASCADE;",
    "DROP TABLE IF EXISTS image CASCADE;",
    "DROP TABLE IF EXISTS settings CASCADE;",
    "DROP TABLE IF EXISTS project CASCADE;"
]
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        for stmt in drop_sql:
            try:
                cur.execute(stmt)
            except Exception as e:
                print(f"Error dropping table: {e}")
print("All tables dropped (if existed).")

All tables dropped (if existed).


## 2. Connect to Citus Node

Establish a connection to the Citus/Postgres node using psycopg. Store connection parameters securely (e.g., using environment variables).

In [6]:
# Use CitusHeadClient for connection
def get_head_connection():
    return CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).get_connection()

# Test connection
with get_head_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        print('Connected to:', cur.fetchone()['version'])

Connected to: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create Database Schema (Tables)

Create all tables as described in the technical design document, including distributed and reference tables. Use Citus distribution commands where required.

In [7]:
# SQL statements for schema creation (reference and distributed tables)
schema_sql = [
    # Project table
    '''CREATE TABLE IF NOT EXISTS project (
        project_id SERIAL PRIMARY KEY,
        project_name TEXT NOT NULL,
        description TEXT
    );''',
    # Image table
    '''CREATE TABLE IF NOT EXISTS image (
        image_id SERIAL PRIMARY KEY,
        project_id INT NOT NULL REFERENCES project(project_id),
        name TEXT NOT NULL,
        image_path TEXT NOT NULL,
        upload_ts TIMESTAMP NOT NULL,
        base_mag FLOAT NOT NULL,
        base_width INT NOT NULL,
        base_height INT NOT NULL,
        deepzoom_tilesize INT NOT NULL,
        embedding_x FLOAT,
        embedding_y FLOAT,
        group_id INT,
        train_test_split INT,
        UNIQUE(project_id, name)
    );''',
    # Label class table
    '''CREATE TABLE IF NOT EXISTS label_class (
        label_class_id SERIAL PRIMARY KEY,
        project_id INT NOT NULL REFERENCES project(project_id),
        name TEXT NOT NULL,
        color_code TEXT,
        event_ts TIMESTAMP NOT NULL,
        UNIQUE(project_id, name)
    );''',
    # Patch table (distributed)
    '''CREATE TABLE IF NOT EXISTS patch (
        patch_id BIGSERIAL PRIMARY KEY,
        patch_uid INT,
        label_class_id SMALLINT NOT NULL REFERENCES label_class(label_class_id),
        image_id INT NOT NULL REFERENCES image(image_id),
        working_mag FLOAT NOT NULL,
        patch_image BYTEA NOT NULL
    );''',
    # Prediction tables (distributed)
    '''CREATE TABLE IF NOT EXISTS pred_patch_latest (
        patch_id BIGINT PRIMARY KEY,
        embed_x FLOAT NOT NULL,
        embed_y FLOAT NOT NULL,
        grid_cell_i SMALLINT NOT NULL,
        grid_cell_j SMALLINT NOT NULL,
        event_ts TIMESTAMP NOT NULL,
        label_class_id SMALLINT NOT NULL REFERENCES label_class(label_class_id)
    );''',
    '''CREATE TABLE IF NOT EXISTS pred_patch_last (
        patch_id BIGINT PRIMARY KEY,
        embed_x FLOAT NOT NULL,
        embed_y FLOAT NOT NULL,
        grid_cell_i SMALLINT NOT NULL,
        grid_cell_j SMALLINT NOT NULL,
        event_ts TIMESTAMP NOT NULL,
        label_class_id SMALLINT NOT NULL REFERENCES label_class(label_class_id)
    );''',
    # Settings table
    '''CREATE TABLE IF NOT EXISTS settings (
        setting_id SERIAL PRIMARY KEY,
        project_id INT REFERENCES project(project_id),
        setting_key TEXT NOT NULL,
        setting_value TEXT NOT NULL,
        disabled BOOLEAN DEFAULT FALSE
    );''',
    # Confusion matrix LN table (distributed)
    '''CREATE TABLE IF NOT EXISTS confusion_matrix_ln (
        shard_id BIGINT NOT NULL,
        grid_cell_i SMALLINT NOT NULL,
        grid_cell_j SMALLINT NOT NULL,
        bucket_date DATE NOT NULL,
        pred_label SMALLINT NOT NULL REFERENCES label_class(label_class_id),
        gt_label SMALLINT NOT NULL REFERENCES label_class(label_class_id),
        count INT NOT NULL,
        PRIMARY KEY (grid_cell_i, grid_cell_j, pred_label, gt_label, shard_id)
    );'''
]

citus_distribution_sql = [
    # Make referenced tables reference tables   (Will need to think about potential performance consequences)
    "SELECT create_reference_table('project');",
    "SELECT create_reference_table('image');",
    "SELECT create_reference_table('label_class');",
    "SELECT create_reference_table('settings');",
    # Distribute tables as required
    "SELECT create_distributed_table('patch', 'patch_id');",
    "SELECT create_distributed_table('pred_patch_latest', 'patch_id');",
    "SELECT create_distributed_table('pred_patch_last', 'patch_id');",
    "SELECT create_distributed_table('confusion_matrix_ln', 'shard_id');"
]

with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        for stmt in schema_sql:
            cur.execute(stmt)
        for stmt in citus_distribution_sql:
            try:
                cur.execute(stmt)
            except Exception as e:
                print(f"Distribution command failed (may already be distributed): {e}")
print("Schema and distribution setup complete.")

Schema and distribution setup complete.


## 4. Verify Table Creation

Query the information schema to verify that all tables have been created successfully.

In [8]:
# List all tables in the public schema
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = [row['table_name'] for row in cur.fetchall()]
        print('Tables in public schema:', tables)

Tables in public schema: ['citus_schemas', 'citus_tables', 'confusion_matrix_ln', 'image', 'label_class', 'patch', 'pred_patch_last', 'pred_patch_latest', 'project', 'settings']


## 5. Insert Dummy Data into Patch Table

Generate and insert dummy data into the patch table, ensuring all required fields are populated and constraints are respected.

In [9]:
# Helper: Insert dummy project, image, label_class for FK constraints
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO project (project_name, description) VALUES (%s, %s) RETURNING project_id;", ('Demo Project', 'For dummy data'))
        project_id = cur.fetchone()['project_id']
        cur.execute("INSERT INTO image (project_id, name, image_path, upload_ts, base_mag, base_width, base_height, deepzoom_tilesize) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) RETURNING image_id;",
                    (project_id, 'Demo Image', '/tmp/demo.tif', datetime.datetime.now(), 20.0, 10000, 8000, 256))
        image_id = cur.fetchone()['image_id']
        cur.execute("INSERT INTO label_class (project_id, name, color_code, event_ts) VALUES (%s, %s, %s, %s) RETURNING label_class_id;",
                    (project_id, 'Tumor', '#FF0000', datetime.datetime.now()))
        label_class_id = cur.fetchone()['label_class_id']
        print(f"Inserted project_id={project_id}, image_id={image_id}, label_class_id={label_class_id}")

def random_bytes(size=128):
    return os.urandom(size)

for i in range(NUM_PATCHES):
    patch_id = head_client.insert_patch(1000 + i, label_class_id, image_id, 20.0, random_bytes())
    print(f"Inserted patch_id={patch_id}")

Inserted project_id=1, image_id=1, label_class_id=1
Inserted patch_id=1
Inserted patch_id=2
Inserted patch_id=3
Inserted patch_id=4
Inserted patch_id=5
Inserted patch_id=6
Inserted patch_id=7
Inserted patch_id=8
Inserted patch_id=9
Inserted patch_id=10
Inserted patch_id=11
Inserted patch_id=12
Inserted patch_id=13
Inserted patch_id=14
Inserted patch_id=15
Inserted patch_id=16
Inserted patch_id=17
Inserted patch_id=18
Inserted patch_id=19
Inserted patch_id=20
Inserted patch_id=21
Inserted patch_id=22
Inserted patch_id=23
Inserted patch_id=24
Inserted patch_id=25
Inserted patch_id=26
Inserted patch_id=27
Inserted patch_id=28
Inserted patch_id=29
Inserted patch_id=30
Inserted patch_id=31
Inserted patch_id=32
Inserted patch_id=33
Inserted patch_id=34
Inserted patch_id=35
Inserted patch_id=36
Inserted patch_id=37
Inserted patch_id=38
Inserted patch_id=39
Inserted patch_id=40
Inserted patch_id=41
Inserted patch_id=42
Inserted patch_id=43
Inserted patch_id=44
Inserted patch_id=45
Inserted pat

In [10]:
# Check number of shards for the patch table and print row counts per shard, including empty shards
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        # Number of shards
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'patch' table: {num_shards}")
        # Row counts per shard, including empty
        cur.execute("""
            SELECT s.shardid, COALESCE(count(p.patch_id), 0) as row_count
            FROM pg_dist_shard s
            LEFT JOIN patch p ON get_shard_id_for_distribution_column('patch', p.patch_id) = s.shardid
            WHERE s.logicalrelid = 'patch'::regclass
            GROUP BY s.shardid
            ORDER BY s.shardid;
        """)
        rows = cur.fetchall()
        empty_count = 0
        for row in rows:
            print(f"Shard {row['shardid']}: {row['row_count']} rows")
            if row['row_count'] == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")

Number of shards for 'patch' table: 32
Shard 102424: 25 rows
Shard 102425: 34 rows
Shard 102426: 30 rows
Shard 102427: 32 rows
Shard 102428: 31 rows
Shard 102429: 31 rows
Shard 102430: 28 rows
Shard 102431: 36 rows
Shard 102432: 29 rows
Shard 102433: 26 rows
Shard 102434: 37 rows
Shard 102435: 35 rows
Shard 102436: 35 rows
Shard 102437: 33 rows
Shard 102438: 33 rows
Shard 102439: 35 rows
Shard 102440: 35 rows
Shard 102441: 24 rows
Shard 102442: 25 rows
Shard 102443: 33 rows
Shard 102444: 33 rows
Shard 102445: 30 rows
Shard 102446: 26 rows
Shard 102447: 32 rows
Shard 102448: 28 rows
Shard 102449: 29 rows
Shard 102450: 20 rows
Shard 102451: 39 rows
Shard 102452: 33 rows
Shard 102453: 32 rows
Shard 102454: 37 rows
Shard 102455: 34 rows
Empty shards: 0 out of 32


## 6. Verify Dummy Data in Patch Table

Query the patch table to confirm that dummy data has been inserted correctly.

In [11]:
# Query and display dummy patch data
for row in head_client.fetch_patches(limit=10):
    print(row)

{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'I\xe2\x8c\xcd$\xceTH\xc5\xf6L<\xaa\xbf\xa7V\xa7\xfd8C\x9f\r\xe1\x18\x96\xb6\xf9\\\xc6\x99\xf8\xd2\x8c\x0b\xf95\x8e`|\xb6Yx\xcbD\xdbmQF\xa4\xe4\xc2\xbf-\x9ds\x9fP\xeeYQQ\xc4\xab\xe0\x98\xbe\x0c\xf6\x08\x07\x8a|t)\xf7\x82D\xc2D\xb1\x97\xe5\xf5\x14\xd8\\\x91\x1a\xa2\x0b;$\xac]\x0b,\x1aV\xbctr\x90\xfd\xd9\x19\n\x8d|.\xbf(\xaa^\xf0@p{\x10\xacP\x8a\xfdi$<\xb6Y\xd4'}
{'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'\xc8\x98\xc6gYfG\xfah\x7f\xd2|\xcd\xa2\x1e\x8c\x94\x0f3v*\xf9/\xe2\xee\xac\xcf\xd4\xe6\x9f\x9b\xe3\x1f1T@\xc4h\xd4\xe9\xb1*v\xea\xde\xb3Q\t\xee\x05\xf0\xcd\xcfp6G\xdbM#\x16\xe7\xe2\n\xed\x95/\x166\x8fT\x08w7+R\x8f\x94JF)\x1f!\xa9\xd8X(\xaf\xef^\x0fH5\r\x8a\xcf\xa4j\xbd\xa6\xcb&\xf0\xbe\x1e\xab\x15\x07\xc2%^\xa6\xf1\xe4+/\xf5\xb4#\x89J\xf9\x9ft\x8bY{$\xf7'}
{'patch_id': 60, 'patch_uid': 1059, 'label_class_id': 1, 'i

## 7. Implement db_client SDK: Head Node Level

Write Python classes and functions in `db_client.py` to interact with the database at the Citus head node level, including connection management and basic CRUD operations.

In [12]:
# db_client.py will be implemented in the next step.
# Example usage for SDK will be shown after SDK implementation.

In [13]:
# Example: Using db_client SDK (single-node)
# Uses constants.py for all connection parameters
head_client = CitusHeadClient()
print('Patches:', head_client.fetch_patches(limit=3))

# Insert a new patch (dummy data)
# patch_id = head_client.insert_patch(2000, 1, 1, 20.0, b'dummybytes')
# print('Inserted patch_id:', patch_id)

Patches: [{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'I\xe2\x8c\xcd$\xceTH\xc5\xf6L<\xaa\xbf\xa7V\xa7\xfd8C\x9f\r\xe1\x18\x96\xb6\xf9\\\xc6\x99\xf8\xd2\x8c\x0b\xf95\x8e`|\xb6Yx\xcbD\xdbmQF\xa4\xe4\xc2\xbf-\x9ds\x9fP\xeeYQQ\xc4\xab\xe0\x98\xbe\x0c\xf6\x08\x07\x8a|t)\xf7\x82D\xc2D\xb1\x97\xe5\xf5\x14\xd8\\\x91\x1a\xa2\x0b;$\xac]\x0b,\x1aV\xbctr\x90\xfd\xd9\x19\n\x8d|.\xbf(\xaa^\xf0@p{\x10\xacP\x8a\xfdi$<\xb6Y\xd4'}, {'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'\xc8\x98\xc6gYfG\xfah\x7f\xd2|\xcd\xa2\x1e\x8c\x94\x0f3v*\xf9/\xe2\xee\xac\xcf\xd4\xe6\x9f\x9b\xe3\x1f1T@\xc4h\xd4\xe9\xb1*v\xea\xde\xb3Q\t\xee\x05\xf0\xcd\xcfp6G\xdbM#\x16\xe7\xe2\n\xed\x95/\x166\x8fT\x08w7+R\x8f\x94JF)\x1f!\xa9\xd8X(\xaf\xef^\x0fH5\r\x8a\xcf\xa4j\xbd\xa6\xcb&\xf0\xbe\x1e\xab\x15\x07\xc2%^\xa6\xf1\xe4+/\xf5\xb4#\x89J\xf9\x9ft\x8bY{$\xf7'}, {'patch_id': 60, 'patch_uid': 1059, 'label_clas

<!-- Worker node logic omitted for single-node setup -->